In [1]:
import os
import io
import logging
import numpy as np
import cv2
from PIL import Image
from scipy.ndimage import gaussian_filter
from pathlib import Path

# ── الإعدادات القياسية للمشروع ──────────────────────────────────────────────
TARGET_SIZE = (256, 256)          # الحجم المعتمد[cite: 2]
ELA_QUALITY = 90                  # جودة ELA المطلوبة[cite: 2]
CANNY_LOW   = 100                 # حد كاني الأدنى[cite: 2]
CANNY_HIGH  = 200                 # حد كاني الأعلى[cite: 2]
GAUSS_SIGMA = 1.0                 
EPSILON     = 1e-8                

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# ── الدوال التقنية (Forensic Logic)[cite: 2] ────────────────────────────────

def _load_rgb(image_path):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((TARGET_SIZE[1], TARGET_SIZE[0]), Image.LANCZOS)
    return np.array(img, dtype=np.uint8)

def _normalize(arr):
    a_min, a_max = arr.min(), arr.max()
    if a_max - a_min < EPSILON:
        return np.zeros_like(arr, dtype=np.float32)
    return ((arr - a_min) / (a_max - a_min)).astype(np.float32)

def compute_ela(image_path):
    """استخراج Error Level Analysis[cite: 2]"""
    original = _load_rgb(image_path).astype(np.float32)
    buf = io.BytesIO()
    Image.fromarray(original.astype(np.uint8)).save(buf, format="JPEG", quality=ELA_QUALITY)
    buf.seek(0)
    recompressed = np.array(Image.open(buf).convert("RGB"), dtype=np.float32)
    ela_map = np.abs(original - recompressed)
    ela_map = np.sqrt(np.sum(ela_map ** 2, axis=2, keepdims=True))
    return _normalize(ela_map).astype(np.float32)

def compute_noise_residual(image):
    """استخراج Noise Residual[cite: 2]"""
    img = image.astype(np.float32) / 255.0
    denoised = np.stack([gaussian_filter(img[:, :, c], sigma=GAUSS_SIGMA) for c in range(3)], axis=2)
    residual = np.abs(img - denoised)
    weights = np.array([0.2126, 0.7152, 0.0722], dtype=np.float32)
    lum_residual = np.sum(residual * weights, axis=2, keepdims=True)
    return _normalize(lum_residual).astype(np.float32)

def compute_edge_map(image):
    """استخراج Edge Map باستخدام Canny[cite: 2]"""
    img_uint8 = image if image.max() > 1.0 else (image * 255).astype(np.uint8)
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, CANNY_LOW, CANNY_HIGH)
    edge_map = edges[:, :, np.newaxis].astype(np.float32) / 255.0
    return _normalize(edge_map).astype(np.float32)

# ── تشغيل المعالجة في كاجل ────────────────────────────────────────────────

def run_processing_on_kaggle(input_dir, output_dir):
    image_dir = Path(input_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    # البحث عن الصور في مجلد الإدخال (Read-only)
    image_paths = sorted([
        p for p in image_dir.rglob("*") 
        if p.suffix.lower() in SUPPORTED_EXTS and "_gt" not in p.name
    ])

    log.info(f"جاري معالجة {len(image_paths)} صورة من مجلد الإدخال...")

    for idx, img_path in enumerate(image_paths):
        stem = img_path.stem
        try:
            rgb = _load_rgb(str(img_path))
            
            # استخراج الميزات الثلاث[cite: 2]
            ela = compute_ela(str(img_path))
            noise = compute_noise_residual(rgb)
            edge = compute_edge_map(rgb)

            # الحفظ في مجلد العمل (Working Directory) ليتم تحويله لاحقاً لـ Dataset
            np.save(out_path / f"{stem}_ela.npy", ela)
            np.save(out_path / f"{stem}_noise.npy", noise)
            np.save(out_path / f"{stem}_edge.npy", edge)

            if (idx + 1) % 100 == 0:
                log.info(f"  [{idx+1}/{len(image_paths)}] تم الانتهاء من {stem}")

        except Exception as e:
            log.error(f"  فشل في {stem}: {e}")

    log.info(f"✅ انتهت المهمة! جميع الملفات محفوظة في: {output_dir}")

# --- تحديد المسارات في كاجل ---
# ملاحظة: تأكد من اسم المجلد الخاص بك في الـ input
KAGGLE_INPUT = "/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2"
KAGGLE_OUTPUT = "/kaggle/working/processed_features"

run_processing_on_kaggle(KAGGLE_INPUT, KAGGLE_OUTPUT)

INFO | جاري معالجة 0 صورة من مجلد الإدخال...
INFO | ✅ انتهت المهمة! جميع الملفات محفوظة في: /kaggle/working/processed_features
